# Notebook 11 — Protocolo Experimental con Participantes Humanos

**Proyecto:** Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad  
**Autor:** Jesús Goenaga Peña  
**Fase:** 5 — Validación con participantes humanos

---

## Estructura del protocolo (duración estimada: 60 minutos)

| Fase | Instrumento | Duración |
|------|-------------|----------|
| 1 | Consentimiento informado + firma electrónica | 5 min |
| 2 | Instrumento AdHoc (datos sociodemográficos y salud) | 10 min |
| 3 | BCSE — Guía de administración + registro de puntuaciones | 10 min |
| 4 | Prueba de salud visual (Snellen + **imágenes anaglifas** + historia) | 10 min |
| 5 | 190 tareas de percepción de profundidad → **App nativa Quest 2** | 30 min |

## Instrucciones para el investigador

- Ejecutar las **Celdas 1 y 2** antes de que el participante vea la pantalla.
- Desde la **Celda 3** en adelante, el participante puede ver la pantalla.
- Usar **Ver → Ocultar código** (`Ctrl+M+L`) para mostrar solo la interfaz.
- Para pruebas piloto usar `PILOT01`, `PILOT02` como ID.

## Confidencialidad

Los datos se almacenan únicamente con ID anonimizado. El mapeo nombre↔ID se guarda en `data/private/`.

## Ground truth validado

GT validado empíricamente: generado en NB08-09, corregido para ilusiones, verificado manualmente (15/15 confirmadas).

---

## Celda 1 — Configuración inicial (solo investigador)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, datetime, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import cv2
import warnings
warnings.filterwarnings('ignore')

BASE_DIR    = '/content/drive/MyDrive/cognitive-depth-model'
SPLITS_DIR  = os.path.join(BASE_DIR, 'data', 'splits')
DATA_DIR    = os.path.join(BASE_DIR, 'data', 'participants')
PRIVATE_DIR = os.path.join(BASE_DIR, 'data', 'private')
RESULTS_DIR = os.path.join(BASE_DIR, 'results', 'human_responses')
for d in [DATA_DIR, PRIVATE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

CSV_VALIDATION = os.path.join(SPLITS_DIR, 'validation_set_final.csv')
df_val = pd.read_csv(CSV_VALIDATION)

print('✓ Drive montado y rutas configuradas.')
print(f'  Set de validación: {len(df_val)} tareas')
print()
print('Participantes registrados:')
existing = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('_session.json')])
pilots = [e for e in existing if 'PILOT' in e]
reales = [e for e in existing if 'PILOT' not in e]
print(f'  Reales: {len(reales)} — Pilotos: {len(pilots)}')
for e in existing:
    print(f'    {e}')

## Celda 2 — Registro del participante (solo investigador)

In [ ]:
# ── SOLO INVESTIGADOR ─────────────────────────────────────
# Para participantes reales: P001, P002, P003
# Para pruebas piloto:       PILOT01, PILOT02

PARTICIPANT_ID = 'P001'   # ← CAMBIAR PARA CADA SESIÓN

FECHA_SESION = datetime.datetime.now().strftime('%Y-%m-%d')
HORA_INICIO  = datetime.datetime.now().strftime('%H:%M:%S')
SESSION_CODE = hashlib.md5(f'{PARTICIPANT_ID}{FECHA_SESION}{HORA_INICIO}'.encode()).hexdigest()[:8].upper()

id_file = os.path.join(DATA_DIR, f'{PARTICIPANT_ID}_session.json')
if os.path.exists(id_file):
    print(f'⚠ Ya existe sesión para {PARTICIPANT_ID}.')
else:
    es_piloto = 'PILOT' in PARTICIPANT_ID
    print(f'✓ ID: {PARTICIPANT_ID}  {"[PILOTO]" if es_piloto else "[PARTICIPANTE REAL]"}')
    print(f'  Fecha: {FECHA_SESION} | Hora: {HORA_INICIO} | Código: {SESSION_CODE}')
    print()
    print('Puedes compartir pantalla y ejecutar la Celda 3.')

session_data = {
    'participant_id': PARTICIPANT_ID, 'es_piloto': 'PILOT' in PARTICIPANT_ID,
    'session_code': SESSION_CODE, 'fecha': FECHA_SESION, 'hora_inicio': HORA_INICIO,
    'consentimiento': {}, 'adhoc': {}, 'bcse': {}, 'salud_visual': {},
}

---
## FASE 1 — Consentimiento Informado
### Celda 3 — Presentación y firma electrónica

In [ ]:
display(HTML(f'''\
<div style="font-family:Arial;max-width:820px;margin:0 auto;padding:20px;">\
<div style="background:#1a5276;color:white;padding:15px;border-radius:8px 8px 0 0;text-align:center;">\
<h2 style="margin:0;">CONSENTIMIENTO INFORMADO</h2>\
<p style="margin:5px 0 0;">Grupo de Investigación: Sistemas Cognitivos Artificiales</p>\
</div>\
<div style="border:2px solid #1a5276;border-top:none;padding:25px;border-radius:0 0 8px 8px;background:#fdfefe;">\
<p style="text-align:right;color:#555;">Manizales, {FECHA_SESION}</p>\
<p><strong>Investigación:</strong> Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad</p>\
<p><strong>Investigador principal:</strong> Jesús Goenaga Peña — Doctorando en Ciencias Cognitivas, UAM</p>\
<p><strong>Director:</strong> Luis Fernando Castillo Ossa (PhD)</p>\
<h3>¿En qué consiste su participación?</h3>\
<p>Sesión única de aproximadamente <strong>60 minutos</strong> con las siguientes actividades:</p>\
<ol>\
<li>Formulario sociodemográfico y auto-reporte de salud (Instrumento AdHoc).</li>\
<li>Test Breve de Evaluación del Estado Cognitivo (BCSE).</li>\
<li>Pruebas de salud visual (agudeza visual e imágenes anaglifas para visión binocular).</li>\
<li>190 tareas de percepción de profundidad mediante dispositivo de realidad virtual (Meta Quest 2).</li>\
</ol>\
<h3>Información importante</h3>\
<ul>\
<li>Participación completamente <strong>libre y voluntaria</strong>. Puede retirarse en cualquier momento.</li>\
<li>Se registran respuestas y tiempos de reacción. <strong>No</strong> hay registro fotográfico ni de video.</li>\
<li>Datos almacenados con <strong>código anónimo</strong>. Su nombre nunca aparece en los archivos.</li>\
<li>Información usada exclusivamente para investigación científica.</li>\
<li><strong>Compensación económica razonable</strong> por su tiempo.</li>\
<li>Contacto: jesus.goenagap@autonoma.edu.co</li>\
</ul>\
<div style="background:#eaf2ff;padding:12px;border-radius:6px;border-left:4px solid #1a5276;margin-top:18px;">\
<strong>Código de sesión:</strong> {SESSION_CODE}<br>\
<small>Consérvelo para solicitar eliminación de datos en el futuro.</small>\
</div></div></div>\
'''))

print('─'*60)
print('FIRMA ELECTRÓNICA')
print('─'*60)
print()
nombre_w = widgets.Text(description='Nombre completo:',
    layout=widgets.Layout(width='500px'), style={'description_width':'160px'})
doc_w = widgets.Text(description='No. documento:',
    layout=widgets.Layout(width='400px'), style={'description_width':'160px'})
acepta_w = widgets.Checkbox(value=False,
    description='He leído y comprendido el consentimiento y acepto participar voluntariamente.',
    layout=widgets.Layout(width='720px'))
btn_firma = widgets.Button(description='Confirmar firma electrónica',
    button_style='success', layout=widgets.Layout(width='280px', height='42px'))
out_firma = widgets.Output()

def on_firma(b):
    with out_firma:
        clear_output()
        if not acepta_w.value:
            print('⚠ Marque la casilla de aceptación.'); return
        if not nombre_w.value.strip() or not doc_w.value.strip():
            print('⚠ Complete todos los campos.'); return
        ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        session_data['consentimiento'] = {
            'nombre_hash': hashlib.sha256(nombre_w.value.strip().encode()).hexdigest(),
            'doc_hash': hashlib.sha256(doc_w.value.strip().encode()).hexdigest(),
            'acepto': True, 'timestamp': ts, 'session_code': SESSION_CODE,
        }
        with open(os.path.join(PRIVATE_DIR, f'{PARTICIPANT_ID}_identity.json'), 'w') as f:
            json.dump({'participant_id': PARTICIPANT_ID, 'nombre': nombre_w.value.strip(),
                       'documento': doc_w.value.strip(), 'timestamp': ts,
                       'session_code': SESSION_CODE}, f, ensure_ascii=False, indent=2)
        print(f'✓ Consentimiento firmado. Timestamp: {ts} | Código: {SESSION_CODE}')
        print()
        print('Ejecute la Celda 4 → Instrumento AdHoc.')

btn_firma.on_click(on_firma)
display(nombre_w, doc_w, acepta_w, btn_firma, out_firma)

---
## FASE 2 — Instrumento AdHoc
### Celda 4 — Datos sociodemográficos y salud

In [ ]:
display(HTML('<h2 style="font-family:Arial;color:#1a5276;border-bottom:2px solid #1a5276;padding-bottom:8px;">📋 INSTRUMENTO AdHoc</h2><p style="font-family:Arial;">Complete los campos. Sus respuestas son confidenciales.</p>'))

edad_w  = widgets.BoundedIntText(description='Edad:', min=18, max=99, value=30,
    layout=widgets.Layout(width='250px'), style={'description_width':'120px'})
genero_w = widgets.RadioButtons(description='Género:', options=['Masculino','Femenino','Otro'],
    layout=widgets.Layout(width='350px'), style={'description_width':'120px'})
educ_w = widgets.Select(description='Nivel educativo:',
    options=['Sin educación formal','Primaria','Bachillerato','Técnico/Tecnológico',
             'Carrera profesional','Especialización','Maestría','Doctorado'],
    layout=widgets.Layout(width='450px'), style={'description_width':'150px'})
ocup_w = widgets.Select(description='Ocupación:',
    options=['Sin empleo','Estudiante','Trabajador independiente','Trabajador dependiente','Otro'],
    layout=widgets.Layout(width='420px'), style={'description_width':'150px'})
civil_w = widgets.Select(description='Estado civil:',
    options=['Soltero/a','Casado/a','Conviviente','Separado/a','Divorciado/a','Viudo/a'],
    layout=widgets.Layout(width='350px'), style={'description_width':'150px'})
display(HTML('<h3 style="font-family:Arial;">Datos sociodemográficos</h3>'))
display(edad_w, genero_w, educ_w, ocup_w, civil_w)

prob_vis_w = widgets.RadioButtons(description='¿Problema visual diagnosticado?',
    options=['Sí','No'], layout=widgets.Layout(width='400px'), style={'description_width':'270px'})
tipo_vis_w = widgets.SelectMultiple(description='Tipo:',
    options=['Miopía','Hipermetropía','Astigmatismo','Presbicia','Otro'],
    layout=widgets.Layout(width='380px'), style={'description_width':'80px'})
correc_w = widgets.RadioButtons(description='Corrección visual:',
    options=['Gafas','Lentes de contacto','Ninguna'],
    layout=widgets.Layout(width='350px'), style={'description_width':'160px'})
examen_w = widgets.RadioButtons(description='Frecuencia exámenes:',
    options=['Regularmente (1-2 años)','De vez en cuando (>2 años)','Nunca'],
    layout=widgets.Layout(width='450px'), style={'description_width':'190px'})
display(HTML('<h3 style="font-family:Arial;">Salud visual</h3>'))
display(prob_vis_w, tipo_vis_w, correc_w, examen_w)

t_cog_w  = widgets.RadioButtons(description='¿Trastorno neurocognitivo diagnosticado?',
    options=['Sí','No'], layout=widgets.Layout(width='450px'), style={'description_width':'310px'})
cambios_w = widgets.SelectMultiple(description='Cambios recientes:',
    options=['Cambios en memoria reciente','Dificultad para concentrarse',
             'Dificultad para resolver problemas','Ninguno'],
    layout=widgets.Layout(width='500px'), style={'description_width':'160px'})
display(HTML('<h3 style="font-family:Arial;">Salud neurocognitiva</h3>'))
display(t_cog_w, cambios_w)

t_psiq_w = widgets.RadioButtons(description='¿Trastorno psiquiátrico diagnosticado?',
    options=['Sí','No'], layout=widgets.Layout(width='450px'), style={'description_width':'310px'})
sint_w = widgets.RadioButtons(description='¿Síntomas psiquiátricos recientes?',
    options=['Sí','No'], layout=widgets.Layout(width='400px'), style={'description_width':'260px'})
trat_w = widgets.RadioButtons(description='¿Tratamiento psiquiátrico último año?',
    options=['Sí','No'], layout=widgets.Layout(width='400px'), style={'description_width':'270px'})
display(HTML('<h3 style="font-family:Arial;">Salud psiquiátrica</h3>'))
display(t_psiq_w, sint_w, trat_w)

btn_adhoc = widgets.Button(description='Guardar AdHoc',
    button_style='primary', layout=widgets.Layout(width='220px', height='42px'))
out_adhoc = widgets.Output()

def on_adhoc(b):
    with out_adhoc:
        clear_output()
        session_data['adhoc'] = {
            'edad': edad_w.value, 'genero': genero_w.value,
            'nivel_educativo': educ_w.value, 'ocupacion': ocup_w.value,
            'estado_civil': civil_w.value, 'problema_visual': prob_vis_w.value,
            'tipo_problema_visual': list(tipo_vis_w.value),
            'correccion_visual': correc_w.value, 'frecuencia_examen': examen_w.value,
            'trastorno_neurocog': t_cog_w.value, 'cambios_cognitivos': list(cambios_w.value),
            'trastorno_psiquiatrico': t_psiq_w.value, 'sintomas_psiquiatricos': sint_w.value,
            'tratamiento_psiquiatrico': trat_w.value,
            'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        }
        print('✓ AdHoc guardado. Ejecute la Celda 5 → BCSE.')

btn_adhoc.on_click(on_adhoc)
display(btn_adhoc, out_adhoc)

---
## FASE 3 — BCSE
### Celda 5 — Guía de administración y registro

> **Para el investigador:** Use el cuaderno de estímulos físico (Pearson). Registre las puntuaciones directas. El sistema calcula los puntajes ponderados.

In [ ]:
display(HTML('<h2 style="font-family:Arial;color:#1a5276;">📊 BCSE — Registro de Puntuaciones</h2>'
             '<div style="background:#eaf2ff;padding:12px;border-radius:6px;border-left:4px solid #1a5276;margin-bottom:15px;">'
             'Registre las puntuaciones directas. El sistema calcula los puntajes ponderados según el manual Pearson.</div>'))

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">1. Orientación (1-5, máx=5)</h3>'))
orient_w = widgets.BoundedIntText(description='Puntuación directa:', min=0, max=5, value=0,
    layout=widgets.Layout(width='300px'), style={'description_width':'200px'})
display(orient_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">2. Estimación temporal (ítem 6)</h3>'))
et_w = widgets.BoundedIntText(description='Diferencia en minutos:', min=0, max=720, value=0,
    layout=widgets.Layout(width='300px'), style={'description_width':'200px'})
display(et_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">3. Control mental (ítems 7-8)</h3>'))
cm_t_w = widgets.BoundedIntText(description='Tiempo (segundos):', min=0, max=300, value=0,
    layout=widgets.Layout(width='280px'), style={'description_width':'180px'})
cm_e_w = widgets.BoundedIntText(description='Errores:', min=0, max=20, value=0,
    layout=widgets.Layout(width='250px'), style={'description_width':'150px'})
display(cm_t_w, cm_e_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">4. Dibujo del reloj (ítem 9, máx=15)</h3>'))
reloj_w = widgets.BoundedIntText(description='Puntuación directa:', min=0, max=15, value=0,
    layout=widgets.Layout(width='300px'), style={'description_width':'200px'})
display(reloj_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">5. Recuerdo incidental (ítem 10)</h3>'
             '<p style="font-family:Arial;font-size:0.9em;">Palabras: silla, gafas, pájaro, taza.</p>'))
recuerdo_w = widgets.BoundedIntText(description='Palabras recordadas:', min=0, max=4, value=0,
    layout=widgets.Layout(width='280px'), style={'description_width':'200px'})
display(recuerdo_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">6. Inhibición (ítem 11)</h3>'))
inh_t_w = widgets.BoundedIntText(description='Tiempo (segundos):', min=0, max=300, value=0,
    layout=widgets.Layout(width='280px'), style={'description_width':'180px'})
inh_o_w = widgets.BoundedIntText(description='Omisiones:', min=0, max=24, value=0,
    layout=widgets.Layout(width='240px'), style={'description_width':'150px'})
inh_c_w = widgets.BoundedIntText(description='Comisiones:', min=0, max=24, value=0,
    layout=widgets.Layout(width='240px'), style={'description_width':'150px'})
display(inh_t_w, inh_o_w, inh_c_w)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">7. Producción verbal (ítem 12, 30 seg)</h3>'))
pv_w = widgets.BoundedIntText(description='Palabras producidas:', min=0, max=50, value=0,
    layout=widgets.Layout(width='280px'), style={'description_width':'200px'})
display(pv_w)

btn_bcse = widgets.Button(description='Calcular y guardar BCSE',
    button_style='primary', layout=widgets.Layout(width='260px', height='42px'))
out_bcse = widgets.Output()

def pond_orient(d): return 0 if d<=3 else (5 if d==4 else 8)
def pond_et(m):     return 0 if m>55 else (1 if m>=31 else (2 if m>=19 else (3 if m>=10 else 4)))
def pond_cm_t(s):   return 0 if s>75 else (1 if s>=56 else (2 if s>=43 else (3 if s>=31 else 4)))
def pond_cm_e(e):   return 0 if e>3 else (2 if e>=2 else (4 if e==1 else 8))
def pond_reloj(d):  return 0 if d<=6 else (1 if d<=8 else (2 if d==9 else (3 if d<=11 else 4)))
def pond_recuerdo(d): return 0 if d==0 else (2 if d==1 else (4 if d==2 else 8))
def pond_inh_t(s):  return 0 if s>48 else (1 if s>=38 else (2 if s>=30 else (3 if s>=23 else 4)))
def pond_inh_o(o):  return 0 if o>=1 else 4
def pond_inh_c(c):
    tabla = [(range(8,25),0),(range(7,8),1),(range(6,7),2),(range(5,6),3),
             (range(4,5),4),(range(3,4),5),(range(2,3),6),(range(1,2),7),(range(0,1),8)]
    for r,p in tabla:
        if c in r: return p
    return 0
def pond_pv(p): return 0 if p<=6 else (1 if p==7 else (2 if p==8 else (4 if p==9 else 6)))

def on_bcse(b):
    with out_bcse:
        clear_output()
        p = [pond_orient(orient_w.value), pond_et(et_w.value),
             pond_cm_t(cm_t_w.value), pond_cm_e(cm_e_w.value),
             pond_reloj(reloj_w.value), pond_recuerdo(recuerdo_w.value),
             pond_inh_t(inh_t_w.value), pond_inh_o(inh_o_w.value),
             pond_inh_c(inh_c_w.value), pond_pv(pv_w.value)]
        total = sum(p)
        session_data['bcse'] = {
            'orientacion_directa': orient_w.value, 'orientacion_ponderada': p[0],
            'et_min': et_w.value, 'et_ponderada': p[1],
            'cm_tiempo_seg': cm_t_w.value, 'cm_tiempo_ponderada': p[2],
            'cm_errores': cm_e_w.value, 'cm_errores_ponderada': p[3],
            'reloj_directa': reloj_w.value, 'reloj_ponderada': p[4],
            'recuerdo_directa': recuerdo_w.value, 'recuerdo_ponderada': p[5],
            'inhibicion_tiempo_seg': inh_t_w.value, 'inhibicion_tiempo_pond': p[6],
            'inhibicion_omisiones': inh_o_w.value, 'inhibicion_omis_pond': p[7],
            'inhibicion_comisiones': inh_c_w.value, 'inhibicion_comis_pond': p[8],
            'produccion_verbal': pv_w.value, 'produccion_verbal_pond': p[9],
            'total_bcse': total, 'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        }
        print(f'✓ BCSE calculado. Total: {total}/58')
        if total >= 47:   print('  → Clasificación: NORMAL')
        elif total >= 40: print('  → Clasificación: NORMAL-BAJO / LÍMITE')
        else:             print('  → Clasificación: BAJO / MUY BAJO (revisar criterio de exclusión)')
        print()
        print('Ejecute la Celda 6 → Prueba de Salud Visual.')

btn_bcse.on_click(on_bcse)
display(btn_bcse, out_bcse)

---
## FASE 4 — Prueba de Salud Visual
### Celda 6 — Agudeza visual + imágenes anaglifas + historia autoinformada

In [ ]:
display(HTML('<h2 style="font-family:Arial;color:#1a5276;border-bottom:2px solid #1a5276;padding-bottom:8px;">\U0001f441 PRUEBA DE SALUD VISUAL</h2>'))

# -- SECCION 1: Agudeza visual -----------------------------------------------
display(HTML('<h3 style="font-family:Arial;color:#2471a3;">Secci\u00f3n 1. Agudeza visual (cartilla Snellen a 3 m)</h3>'))
display(HTML('<p style="font-family:Arial;font-size:0.9em;">Evaluar un ojo a la vez. Registrar el nivel m\u00e1s alto alcanzado sin errores.</p>'))
oi_w = widgets.Select(description='Ojo izquierdo:',
    options=['20/20','20/25','20/30','20/40','Peor de 20/40','No evaluable'],
    layout=widgets.Layout(width='340px'), style={'description_width':'130px'})
od_w = widgets.Select(description='Ojo derecho:',
    options=['20/20','20/25','20/30','20/40','Peor de 20/40','No evaluable'],
    layout=widgets.Layout(width='340px'), style={'description_width':'130px'})
cor_w = widgets.RadioButtons(description='\u00bfUs\u00f3 correcci\u00f3n?',
    options=['S\u00ed (gafas)','S\u00ed (lentes de contacto)','No'],
    layout=widgets.Layout(width='360px'), style={'description_width':'140px'})
display(oi_w, od_w, cor_w)

# -- SECCION 2: Vision binocular con imagenes anaglifas ----------------------
from io import BytesIO
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage

def _crear_anaglifo(escena):
    """
    Genera imagen anaglifa rojo/cian para test de vision binocular.
    Canal R = ojo izquierdo (desplazado +disparity/2 hacia la derecha)
    Canal G,B = ojo derecho (desplazado -disparity/2 hacia la izquierda)
    Con gafas rojo/cian: mayor disparity -> objeto parece mas CERCANO.
    """
    H, W = 300, 600
    cL = np.full((H, W), 230, dtype=np.float32)
    cR = np.full((H, W), 230, dtype=np.float32)
    Y, X = np.mgrid[:H, :W]
    label_info = []
    for obj in escena['objetos']:
        cx, cy = obj['cx'], obj['cy']
        d = obj['disparity']
        cxL = cx + d // 2
        cxR = cx - d // 2
        if obj['forma'] == 'circulo':
            r = obj['radio']
            mL = (X - cxL)**2 + (Y - cy)**2 <= r**2
            mR = (X - cxR)**2 + (Y - cy)**2 <= r**2
            ly = cy + r + 22
        else:
            w2, h2 = obj['ancho'] // 2, obj['alto'] // 2
            mL = (np.abs(X - cxL) <= w2) & (np.abs(Y - cy) <= h2)
            mR = (np.abs(X - cxR) <= w2) & (np.abs(Y - cy) <= h2)
            ly = cy + h2 + 22
        cL[mL] = obj['color']
        cR[mR] = obj['color']
        label_info.append({'l': obj['label'], 'x': cx, 'y': min(int(ly), H - 18)})
    anag = np.zeros((H, W, 3), dtype=np.uint8)
    anag[:, :, 0] = np.clip(cL, 0, 255).astype(np.uint8)
    anag[:, :, 1] = np.clip(cR, 0, 255).astype(np.uint8)
    anag[:, :, 2] = np.clip(cR, 0, 255).astype(np.uint8)
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.imshow(anag, aspect='auto')
    for li in label_info:
        ax.text(li['x'], li['y'], li['l'], color='white', fontsize=20,
                fontweight='bold', ha='center', va='center',
                bbox=dict(facecolor='#333333', alpha=0.88, pad=4, boxstyle='round'))
    ax.axis('off')
    ax.set_title(escena['descripcion'], fontsize=10, fontweight='bold', pad=10)
    plt.tight_layout(pad=0.4)
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight', facecolor='#eeeeee')
    plt.close(fig)
    buf.seek(0)
    return buf.read()

_ESCENAS = [
    {'id': 1,
     'descripcion': '\u00bfCu\u00e1l objeto parece M\u00c1S CERCANO: A (izquierda) o B (derecha)?',
     'respuesta_correcta': 'A',
     'objetos': [
         {'forma': 'circulo',    'label': 'A', 'cx': 175, 'cy': 135, 'radio': 65, 'disparity': 22, 'color': 70},
         {'forma': 'circulo',    'label': 'B', 'cx': 420, 'cy': 135, 'radio': 65, 'disparity':  4, 'color': 70},
     ]},
    {'id': 2,
     'descripcion': '\u00bfCu\u00e1l objeto parece M\u00c1S CERCANO: A (arriba) o B (abajo)?',
     'respuesta_correcta': 'B',
     'objetos': [
         {'forma': 'rectangulo', 'label': 'A', 'cx': 300, 'cy':  85, 'ancho': 130, 'alto':  80, 'disparity':  4, 'color': 70},
         {'forma': 'rectangulo', 'label': 'B', 'cx': 300, 'cy': 215, 'ancho': 130, 'alto':  80, 'disparity': 22, 'color': 70},
     ]},
    {'id': 3,
     'descripcion': '\u00bfCu\u00e1l c\u00edrculo parece M\u00c1S CERCANO: A (izq), B (centro) o C (der)?',
     'respuesta_correcta': 'B',
     'objetos': [
         {'forma': 'circulo', 'label': 'A', 'cx': 115, 'cy': 135, 'radio': 50, 'disparity':  4, 'color': 70},
         {'forma': 'circulo', 'label': 'B', 'cx': 300, 'cy': 135, 'radio': 50, 'disparity': 24, 'color': 70},
         {'forma': 'circulo', 'label': 'C', 'cx': 485, 'cy': 135, 'radio': 50, 'disparity':  4, 'color': 70},
     ]},
    {'id': 4,
     'descripcion': '\u00bfCu\u00e1l objeto parece M\u00c1S CERCANO: A (c\u00edrculo) o B (rect\u00e1ngulo)?',
     'respuesta_correcta': 'A',
     'objetos': [
         {'forma': 'circulo',    'label': 'A', 'cx': 175, 'cy': 135, 'radio': 65,  'disparity': 22, 'color':  60},
         {'forma': 'rectangulo', 'label': 'B', 'cx': 420, 'cy': 135, 'ancho': 130, 'alto': 120, 'disparity':  4, 'color': 100},
     ]},
    {'id': 5,
     'descripcion': '\u00bfCu\u00e1l objeto parece M\u00c1S CERCANO: A (izquierda) o B (derecha)?',
     'respuesta_correcta': 'B',
     'objetos': [
         {'forma': 'rectangulo', 'label': 'A', 'cx': 175, 'cy': 135, 'ancho':  80, 'alto':  80, 'disparity':  4, 'color': 70},
         {'forma': 'rectangulo', 'label': 'B', 'cx': 420, 'cy': 135, 'ancho': 145, 'alto': 110, 'disparity': 24, 'color': 70},
     ]},
]

_res_anag         = {}
_idx_anag         = [0]
_n_anag_correctos = [0]
_out_anag  = widgets.Output(layout=widgets.Layout(width='100%'))
_lbl_anag  = widgets.HTML(value='')
_btn_ok    = widgets.Button(description='\u2713  Correcto',   button_style='success',
                            layout=widgets.Layout(width='180px', height='48px'))
_btn_mal   = widgets.Button(description='\u2717  Incorrecto', button_style='danger',
                            layout=widgets.Layout(width='180px', height='48px'))

def _mostrar_anag(idx):
    if idx >= len(_ESCENAS):
        n_ok = sum(1 for v in _res_anag.values() if v)
        _n_anag_correctos[0] = n_ok
        _btn_ok.disabled = _btn_mal.disabled = True
        color = '#d4edda' if n_ok >= 3 else '#f8d7da'
        icono = '\u2713' if n_ok >= 3 else '\u2717'
        nota  = ('Estereopsia funcional \u2014 puede continuar.' if n_ok >= 3 else
                 'Estereopsia insuficiente \u2014 revisar criterio de exclusi\u00f3n.')
        with _out_anag:
            clear_output()
            display(HTML(
                f'<div style="background:{color};padding:14px;border-radius:8px;font-family:Arial;">'
                f'<strong>{icono} \u00cdtems correctos: {n_ok} / 5</strong> \u2014 {nota}</div>'
            ))
        _lbl_anag.value = ''
        return
    esc = _ESCENAS[idx]
    _lbl_anag.value = (
        f'<p style="font-family:Arial;font-weight:bold;color:#1a5276;margin:4px 0;">'
        f'Imagen {idx + 1} de {len(_ESCENAS)}</p>'
    )
    with _out_anag:
        clear_output(wait=True)
        display(IPImage(data=_crear_anaglifo(esc), format='png'))

def _on_ok(b):
    _res_anag[_idx_anag[0]] = True
    _idx_anag[0] += 1
    _mostrar_anag(_idx_anag[0])

def _on_mal(b):
    _res_anag[_idx_anag[0]] = False
    _idx_anag[0] += 1
    _mostrar_anag(_idx_anag[0])

_btn_ok.on_click(_on_ok)
_btn_mal.on_click(_on_mal)

display(HTML('<h3 style="font-family:Arial;color:#2471a3;">Secci\u00f3n 2. Visi\u00f3n binocular y disparidad estereosc\u00f3pica</h3>'))
display(HTML(
    '<div style="background:#fff3cd;padding:11px;border-radius:6px;border-left:4px solid #ffc107;margin-bottom:10px;font-family:Arial;">'
    '<strong>\u26a0 Instrucci\u00f3n para el investigador:</strong> Entregue las gafas rojo/cian al participante. '
    'Se mostrar\u00e1n 5 im\u00e1genes una a la vez. Para cada imagen, el participante indica cu\u00e1l objeto '
    'percibe como m\u00e1s cercano. Marque <em>Correcto</em> o <em>Incorrecto</em> seg\u00fan corresponda.'
    '</div>'
))
display(_lbl_anag, _out_anag, widgets.HBox([_btn_ok, widgets.Label('     '), _btn_mal]))
_mostrar_anag(0)

# -- SECCION 3: Historia visual autoinformada ---------------------------------
display(HTML('<h3 style="font-family:Arial;color:#2471a3;">Secci\u00f3n 3. Historia visual autoinformada</h3>'))
_preg_vis = [
    '\u00bfUsa actualmente gafas o lentes de contacto?',
    '\u00bfLe han diagnosticado miop\u00eda, astigmatismo, hipermetrop\u00eda o estrabismo?',
    '\u00bfHa sido sometido a cirug\u00eda ocular (l\u00e1ser, cataratas, etc.)?',
    '\u00bfHa tenido visi\u00f3n doble o dificultad para enfocar en los \u00faltimos 6 meses?',
    '\u00bfTiene antecedentes de enfermedades neurol\u00f3gicas que afecten la visi\u00f3n?',
    '\u00bfLe cuesta seguir objetos en movimiento o detectar diferencias de profundidad?',
]
hist_ws = []
for _p in _preg_vis:
    _w = widgets.RadioButtons(description=_p, options=['S\u00ed', 'No', 'No sabe'],
         layout=widgets.Layout(width='720px'), style={'description_width':'510px'})
    hist_ws.append(_w)
    display(_w)

btn_vis = widgets.Button(description='Guardar y evaluar salud visual',
    button_style='primary', layout=widgets.Layout(width='290px', height='42px'))
out_vis = widgets.Output()

def cls_agudeza(v):
    return ('adecuada'   if v in ['20/20', '20/25'] else
            'sospechosa' if v in ['20/30', '20/40'] else 'limitada')

def on_vis(b):
    with out_vis:
        clear_output()
        if _idx_anag[0] < len(_ESCENAS):
            print('\u26a0 Debe completar las 5 im\u00e1genes anaglifas antes de guardar.')
            return
        ci   = cls_agudeza(oi_w.value)
        cd   = cls_agudeza(od_w.value)
        n_si = sum(1 for _w in hist_ws if _w.value == 'S\u00ed')
        n_ok = _n_anag_correctos[0]
        excl_ag  = (ci == 'limitada' and cd == 'limitada')
        excl_est = (n_ok < 3)
        revision = (n_si >= 3)
        session_data['salud_visual'] = {
            'ojo_izquierdo':         oi_w.value,
            'clase_oi':              ci,
            'ojo_derecho':           od_w.value,
            'clase_od':              cd,
            'correccion_usada':      cor_w.value,
            'estereopsia_correctos': n_ok,
            'resultados_anaglifas':  {str(k): v for k, v in _res_anag.items()},
            'historia':              [_w.value for _w in hist_ws],
            'n_afirmativos':         n_si,
            'exclusion_agudeza':     excl_ag,
            'exclusion_estereopsia': excl_est,
            'revision_adicional':    revision,
            'timestamp':             datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        }
        print('\u2713 Prueba de salud visual guardada.')
        print(f'  OI: {oi_w.value} ({ci}) | OD: {od_w.value} ({cd})')
        print(f'  Estereopsia: {n_ok}/5 \u00edtems correctos')
        print(f'  Historia: {n_si} respuestas afirmativas')
        print()
        if   excl_ag:   print('  \u26d4 EXCLUSI\u00d3N: Agudeza visual limitada en ambos ojos.')
        elif excl_est:  print('  \u26d4 EXCLUSI\u00d3N: Estereopsia insuficiente (< 3/5 im\u00e1genes anaglifas).')
        elif revision:  print('  \u26a0 Participaci\u00f3n condicionada: revisar historia visual con experto.')
        else:           print('  \u2713 Participante apto para las tareas experimentales.')
        print()
        print('Ejecute la Celda 7 \u2192 Guardado de fases 1\u20134 y transici\u00f3n a app Quest 2.')

btn_vis.on_click(on_vis)
display(btn_vis, out_vis)


---
## Celda 7 — Guardado parcial (Fases 1–4) y transición a app Quest 2

In [ ]:
# Ejecutar DESPUÉS de completar la Fase 4 (Prueba de Salud Visual)

hora_fase4 = datetime.datetime.now().strftime('%H:%M:%S')
session_data['hora_fin_fases_1_4'] = hora_fase4

fases_ok = {
    'Consentimiento': bool(session_data.get('consentimiento')),
    'AdHoc':          bool(session_data.get('adhoc')),
    'BCSE':           bool(session_data.get('bcse')),
    'Salud visual':   bool(session_data.get('salud_visual')),
}

print('='*60)
print('GUARDADO PARCIAL — Fases 1 a 4')
print('='*60)
for fase, ok in fases_ok.items():
    print(f'  {"✓" if ok else "⚠"} {fase}')
print()

if not all(fases_ok.values()):
    print('⚠ Hay fases incompletas. Complételas antes de continuar.')
else:
    ruta_sesion = os.path.join(DATA_DIR, f'{PARTICIPANT_ID}_session.json')
    with open(ruta_sesion, 'w', encoding='utf-8') as f:
        json.dump(session_data, f, ensure_ascii=False, indent=2)
    print(f'✓ Datos de fases 1–4 guardados en:')
    print(f'  → {ruta_sesion}')
    print()

    sv = session_data.get('salud_visual', {})
    print(f'Participante:       {PARTICIPANT_ID}')
    print(f'BCSE total:         {session_data["bcse"].get("total_bcse", "N/A")}/58')
    print(f'Estereopsia:        {sv.get("estereopsia_correctos", "N/A")}/5 imágenes anaglifas')
    print(f'Exclusión agudeza:  {sv.get("exclusion_agudeza", "N/A")}')
    print(f'Exclusión estereop: {sv.get("exclusion_estereopsia", "N/A")}')
    print()

    excl = sv.get('exclusion_agudeza', False) or sv.get('exclusion_estereopsia', False)
    if excl:
        print('⛔ PARTICIPANTE EXCLUIDO — no continuar con las tareas.')
    else:
        print('✓ Participante apto. Continuar con la App Quest 2:')
        print()
        print('  1. Colocar las gafas Meta Quest 2 al participante.')
        print('  2. Abrir la app "Depth Perception Study" en el Quest 2.')
        print(f'  3. Ingresar el ID del participante: {PARTICIPANT_ID}')
        print('  4. La app guarda automáticamente las respuestas en Drive.')
        print('  5. Al finalizar las 190 tareas, retirar el Quest 2.')
        print()
        print('ℹ Los datos de las 190 tareas se analizan en el NB12.')